# 03 — Model Selection and Evaluation

Continuous-regression selection only. The authenticated preparation handoff is the sole scientific input; the held-out partition remains a structural integrity reference and is never loaded into a modeling variable.

## 1. Model-Selection Context and Boundary

This notebook tunes on train, freezes one configuration per family, then evaluates those frozen candidates once on validation. Final training belongs to Notebook 04.

In [1]:
from pathlib import Path
import json, subprocess, sys, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import cross_validate
from scripts.project_context import get_project_context
from scripts.prepare_data import load_and_validate_preparation_for_model_selection
from scripts.select_models import *


## 2. Independent Preparation-Handoff Loading

In [2]:
PROJECT = get_project_context()
DATASET_SLUG = 'concrete-compressive-strength'
PREPARATION_HANDOFF_PATH = Path('artifacts/preparation/concrete-compressive-strength/preparation-handoff.json')
MODEL_SELECTION_ROOT = Path('artifacts/model-selection/concrete-compressive-strength')
preparation = load_and_validate_preparation_for_model_selection(project_root=PROJECT.root, preparation_handoff_path=PREPARATION_HANDOFF_PATH)
train_frame = preparation.train
validation_frame = preparation.validation
manifests = preparation.manifests
feature_manifest = manifests['feature_manifest']; split_manifest = manifests['split_manifest']
preparation_manifest = manifests['preparation_manifest']; quality_evidence = manifests['quality_evidence']
preparation_handoff = manifests['preparation_handoff']
PREPARED_INTEGRITY_REFERENCE = preparation.prepared_integrity_reference
INTEGRITY_REFERENCE = preparation.sealed_test_integrity_reference
del preparation
assert preparation_handoff['schema_version'] == 'preparation-handoff.v1'
assert feature_manifest['schema_version'] == 'feature-manifest.v3' and split_manifest['schema_version'] == 'split-manifest.v3'
assert preparation_handoff['readiness']['educational_model_selection_ready'] is True


## 3. Frozen Feature, Target, and Partition Roles

In [3]:
TARGET_COLUMN = 'Concrete compressive strength'
FEATURE_COLUMNS = tuple(feature_manifest['feature_columns'])
IDENTIFIER_COLUMNS = tuple(feature_manifest['identifier_columns'])
assert feature_manifest['problem_type'] == 'continuous_regression'
assert feature_manifest['target_contract']['semantics'] == 'Continuous / quantitative'
assert feature_manifest['target_contract']['unit'] == 'MPa'
assert split_manifest['row_counts'] == {'train': 721, 'validation': 154, 'test': 155}
roles = validate_regression_feature_partition_roles(train=train_frame, validation=validation_frame, feature_columns=FEATURE_COLUMNS, identifier_columns=IDENTIFIER_COLUMNS, target_column=TARGET_COLUMN)
X_train, y_train = roles.x_train, roles.y_train
X_validation, y_validation = roles.x_validation, roles.y_validation
del roles, train_frame, validation_frame


## 4. Continuous Regression Metric and CV Contract

In [4]:
PRACTICAL_MAE_TIE_TOLERANCE_MPA = 0.10
CONTRACT = validate_regression_model_selection_contract({'problem_type':'continuous_regression','target_semantics':'Continuous / quantitative','target_unit':'MPa','primary_metric':'mae','primary_metric_direction':'lower_is_better','refit_metric':'mae','cv':{'strategy':'KFold','n_splits':5,'shuffle':True,'random_state':42},'test_partition_sealed':True,'test_partition_evaluated':False})
SCORING = build_regression_scoring_contract(); CV = build_regression_cross_validation()
FOLD_DIAGNOSTICS = describe_regression_cv_folds(cv=CV, x=X_train, y=y_train)


## 5. Candidate Families, Baseline, and Fold-Safe Pipelines

The median dummy aligns with MAE. Ridge scales inside its pipeline; tree families do not scale. All eight prepared predictors remain in the official policy.

In [5]:
BASELINE = build_candidate_pipeline(estimator=DummyRegressor(strategy='median'), numerical_features=FEATURE_COLUMNS, categorical_features=(), scale_numerical=False)
SPECS = [
 {'model_id':'ridge','family':'Ridge','estimator':Ridge(),'scale':True,'space':{'model__alpha':[0.1,1.0,10.0,100.0]}},
 {'model_id':'decision_tree','family':'DecisionTreeRegressor','estimator':DecisionTreeRegressor(random_state=42),'scale':False,'space':{'model__max_depth':[3,5,8,None],'model__min_samples_leaf':[1,5,10]}},
 {'model_id':'random_forest','family':'RandomForestRegressor','estimator':RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=1),'scale':False,'space':{'model__max_depth':[None,12],'model__min_samples_leaf':[1,3,5],'model__max_features':[1.0,'sqrt']}},
 {'model_id':'hist_gradient_boosting','family':'HistGradientBoostingRegressor','estimator':HistGradientBoostingRegressor(max_iter=300,random_state=42),'scale':False,'space':{'model__learning_rate':[0.05,0.10],'model__max_leaf_nodes':[15,31],'model__min_samples_leaf':[10,20,40],'model__l2_regularization':[0.0,1.0]}}
]
assert sum(np.prod([len(v) for v in spec['space'].values()]) for spec in SPECS) == 52


## 6. Train-Only Baseline and Family Search

All hyperparameter decisions below use only the 721 training rows and the frozen five folds.

In [6]:
dummy_raw = cross_validate(BASELINE, X_train.copy(), y_train.copy(), scoring=SCORING, cv=CV, n_jobs=1, return_train_score=False)
DUMMY_CV = {}
for metric in ('mae','rmse','r2','medae'):
    values = np.asarray(dummy_raw['test_'+metric]); values = values if metric == 'r2' else -values
    DUMMY_CV['cv_'+metric+'_mean'] = float(values.mean()); DUMMY_CV['cv_'+metric+'_std'] = float(values.std(ddof=0))
SEARCH_OUTCOMES={}; FAMILY_SUMMARIES={}; tables=[]; WARNINGS={}
for spec in SPECS:
    pipeline=build_candidate_pipeline(estimator=spec['estimator'],numerical_features=FEATURE_COLUMNS,categorical_features=(),scale_numerical=spec['scale'])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        outcome=run_regression_model_search(model_id=spec['model_id'],family=spec['family'],pipeline=pipeline,x_train=X_train,y_train=y_train,scoring=SCORING,cv=CV,search_space=spec['space'],n_jobs=1,error_score='raise')
    summary,table=summarize_regression_search_results(outcome)
    SEARCH_OUTCOMES[spec['model_id']]=outcome; FAMILY_SUMMARIES[spec['model_id']]=summary; tables.append(table)
    WARNINGS[spec['model_id']]=sorted({f'{w.category.__name__}: {w.message}' for w in caught})
CV_RESULTS=pd.concat(tables,ignore_index=True)


## 7. Frozen Family Winners

Grid-search winners are now immutable. No family, metric, grid, feature, or preprocessing choice changes after this point.

In [7]:
FROZEN_PIPELINES={model_id: clone(outcome.best_estimator) for model_id,outcome in SEARCH_OUTCOMES.items()}
FROZEN_PARAMETERS={model_id: outcome.best_parameters for model_id,outcome in SEARCH_OUTCOMES.items()}


## 8. Validation Evaluation

Each frozen candidate and the baseline is fitted on full train, then evaluated once on validation.

In [8]:
BASELINE.fit(X_train,y_train); DUMMY_VALIDATION=evaluate_regression_estimator(estimator=BASELINE,x=X_validation,y_true=y_validation)
VALIDATION={}; FITTED={}
for model_id in sorted(FROZEN_PIPELINES):
    fitted=FROZEN_PIPELINES[model_id]; fitted.fit(X_train,y_train); FITTED[model_id]=fitted
    VALIDATION[model_id]=evaluate_regression_estimator(estimator=fitted,x=X_validation,y_true=y_validation)
SELECTION=select_regression_candidate_model(cv_summaries=FAMILY_SUMMARIES,validation_evaluations=VALIDATION,baseline_validation_metrics=DUMMY_VALIDATION['metrics'],practical_tie_tolerance=PRACTICAL_MAE_TIE_TOLERANCE_MPA)
SELECTED_MODEL_ID=SELECTION['selected_model_id']; SELECTED_SPEC=next(s for s in SPECS if s['model_id']==SELECTED_MODEL_ID)
SELECTED_VALIDATION=VALIDATION[SELECTED_MODEL_ID]


## 9. Regression Error Diagnostics

Only aggregate errors are persisted; row-level predictions and residuals remain transient.

In [9]:
validation_summary=pd.DataFrame([{'model_id':'dummy_median',**DUMMY_VALIDATION['metrics']}]+[{'model_id':m,**e['metrics']} for m,e in VALIDATION.items()])
validation_summary


,model_id,mae,rmse,r2,medae,residual_mean,residual_standard_deviation,max_absolute_error,absolute_error_p50,absolute_error_p90,absolute_error_p95
0,dummy_median,12.814481,15.869853,-0.000494,11.260000,-0.352792,15.917696,41.720000,11.260000,25.162000,28.187000
1,decision_tree,4.285974,6.524744,0.830880,2.900000,0.752338,6.502371,26.340000,2.900000,10.569000,14.749000
2,hist_gradient_boosting,2.741655,4.087048,0.933643,1.803286,0.237051,4.093480,16.799181,1.803286,5.942626,8.468055
3,random_forest,3.768930,5.414638,0.883532,2.538517,0.616447,5.396984,20.666400,2.538517,8.507243,11.600798
4,ridge,8.594383,10.753295,0.540641,7.167060,-0.976668,10.743789,27.871960,7.167060,17.554117,21.478374


## 10. Candidate Selection and Deterministic Tie-Break

Eligibility requires strict validation-MAE improvement over the dummy. Candidates within 0.10 MPa of best MAE use RMSE, MedAE, CV-MAE standard deviation, R², then stable ID.

In [10]:
SELECTION


{'ranking': [{'model_id': 'hist_gradient_boosting',
   'family': 'HistGradientBoostingRegressor',
   'validation_mae': 2.741655333532888,
   'validation_rmse': 4.087048247290223,
   'validation_medae': 1.803285997936193,
   'validation_r2': 0.9336427445907192,
   'cv_mae_std': 0.28291333949152325,
   'eligible': np.True_,
   'absolute_mae_improvement_over_baseline': 10.072825185947632,
   'relative_mae_improvement_percent': 78.60502164434185},
  {'model_id': 'random_forest',
   'family': 'RandomForestRegressor',
   'validation_mae': 3.7689297526283267,
   'validation_rmse': 5.414638236004075,
   'validation_medae': 2.538516666666588,
   'validation_r2': 0.8835316890332094,
   'cv_mae_std': 0.5188042859679992,
   'eligible': np.True_,
   'absolute_mae_improvement_over_baseline': 9.045550766852195,
   'relative_mae_improvement_percent': 70.58850925015014},
  {'model_id': 'decision_tree',
   'family': 'DecisionTreeRegressor',
   'validation_mae': 4.285974025974026,
   'validation_rmse': 6

## 11. Repeated-Profile Sensitivity

In [11]:
REPEATED_SENSITIVITY=analyze_regression_repeated_profile_sensitivity(train_features=X_train,validation_features=X_validation,y_validation=y_validation,predictions=SELECTED_VALIDATION['predictions'])


## 12. Target-Extreme Sensitivity

In [12]:
EXTREME_SENSITIVITY=analyze_regression_target_extreme_sensitivity(y_train=y_train,y_validation=y_validation,predictions=SELECTED_VALIDATION['predictions'])


## 13. Exploratory Hypothesis Conclusions

HYP-REG-001: flexible nonlinear/interacting families may improve predictive error over the additive linear benchmark. Its status is derived only after train CV and frozen validation evaluation.

In [13]:
ridge_mae=VALIDATION['ridge']['metrics']['mae']; flexible_best=min(VALIDATION[m]['metrics']['mae'] for m in ('decision_tree','random_forest','hist_gradient_boosting'))
HYPOTHESIS={'hypothesis_id':'HYP-REG-001','statement':'Flexible nonlinear/interacting model families may improve predictive error over the additive linear benchmark.','status':'supported' if flexible_best < ridge_mae else 'not_supported','evidence':{'ridge_validation_mae':ridge_mae,'best_flexible_validation_mae':flexible_best}}


## 14. Artifact Materialization

In [14]:
prep_hashes={'prepared_sha256':preparation_manifest['prepared_sha256'],'train_sha256':split_manifest['partition_sha256']['train'],'validation_sha256':split_manifest['partition_sha256']['validation'],'test_sha256_integrity_reference_only':split_manifest['partition_sha256']['test'],'preparation_manifest_sha256':preparation_handoff['components']['preparation_manifest']['sha256'],'feature_manifest_sha256':preparation_handoff['components']['feature_manifest']['sha256'],'split_manifest_sha256':preparation_handoff['components']['split_manifest']['sha256'],'quality_evidence_sha256':preparation_handoff['components']['quality_evidence']['sha256']}
target_contract={'column':TARGET_COLUMN,'semantics':'Continuous / quantitative','unit':'MPa','prediction_output':'continuous numeric value on the original target scale'}
metric_contract={'name':'mae','unit':'MPa','direction':'lower_is_better'}
manifest={'schema_version':'model-selection-manifest.v3','artifact_type':'model_selection_manifest','dataset_slug':DATASET_SLUG,'problem_type':'continuous_regression','preparation_handoff_reference':{'path':PREPARATION_HANDOFF_PATH.as_posix(),'schema_version':'preparation-handoff.v1','sha256':sha256_file(PROJECT.path(PREPARATION_HANDOFF_PATH))},'preparation_artifact_hashes':prep_hashes,'target_contract':target_contract,'feature_contract':{'available_features':list(FEATURE_COLUMNS),'selected_features':list(FEATURE_COLUMNS),'selected_feature_policy':'all_features'},'model_selection_contract':{'primary_metric':'mae','primary_metric_unit':'MPa','primary_metric_direction':'lower_is_better','practical_tie_tolerance_mpa':0.10,'secondary_metrics':['rmse','r2','medae']},'candidate_families':[{'model_id':s['model_id'],'family':s['family'],'search_space':s['space'],'scaling':s['scale']} for s in SPECS],'cv_contract':{'strategy':'KFold','n_splits':5,'shuffle':True,'random_state':42,'fit_partition':'train_only'},'search_contract':{'validation_in_search':False,'test_in_search':False},'random_seeds':{'cv':42,'estimators':42},'artifact_paths':{n:(MODEL_SELECTION_ROOT/n).as_posix() for n in REGRESSION_ARTIFACT_FILENAMES},'runtime_versions':runtime_versions(),'test_partition_sealed':True,'test_partition_evaluated':False,'final_model_trained':False,'model_artifact_materialized':False,'model_bundle_materialized':False,'operational_validity':'unconfirmed','limitations':['educational shuffled snapshot','operational validity unconfirmed']}
family_searches=[]
for s in SPECS:
    mid=s['model_id']; family_searches.append({'model_id':mid,'family':s['family'],'search_strategy':'GridSearchCV','search_space':s['space'],'selected_hyperparameters':FROZEN_PARAMETERS[mid],'cv_aggregates':FAMILY_SUMMARIES[mid],'validation_aggregates':VALIDATION[mid]['metrics'],'warnings':WARNINGS[mid],'failures':[]})
candidate_results={'schema_version':'candidate-results.v3','artifact_type':'candidate_results','dataset_slug':DATASET_SLUG,'problem_type':'continuous_regression','primary_metric_contract':metric_contract,'baseline':{'model_id':'dummy_median','family':'DummyRegressor','strategy':'median','cv_aggregates':DUMMY_CV,'validation_aggregates':DUMMY_VALIDATION['metrics']},'family_searches':family_searches,'frozen_candidates':SELECTION['ranking'],'selection':SELECTION,'test_partition_evaluated':False}
validation_evidence={'schema_version':'validation-evidence.v3','artifact_type':'validation_evidence','dataset_slug':DATASET_SLUG,'problem_type':'continuous_regression','partition':'validation','target_contract':target_contract,'primary_metric':metric_contract,'baseline':DUMMY_VALIDATION['metrics'],'models':{m:e['metrics'] for m,e in VALIDATION.items()},'selection_reference':{'selected_model_id':SELECTED_MODEL_ID},'test_partition_evaluated':False}
selection_analysis={'schema_version':'selection-analysis.v3','artifact_type':'selection_analysis','dataset_slug':DATASET_SLUG,'problem_type':'continuous_regression','feature_policy_analysis':{'selected_policy':'all_features','ablation_performed':False,'all_features_only':True,'no_ablation_rationale':['no high-redundancy pair detected','no source-documented derived dependency','all eight validated predictors retained by preparation contract']},'nonlinearity_interaction_hypothesis':HYPOTHESIS,'repeated_profile_sensitivity':REPEATED_SENSITIVITY,'target_extreme_sensitivity':EXTREME_SENSITIVITY,'tie_analysis':{'tolerance_mpa':0.10,'practical_tie':SELECTION['practical_tie'],'finalists':SELECTION['finalists'],'criteria_applied':SELECTION['criteria_applied']},'selection_rationale':SELECTION['selection_rationale'],'test_partition_evaluated':False}
selected_summary=FAMILY_SUMMARIES[SELECTED_MODEL_ID]
preprocessing={'type':'Pipeline','numerical_features':list(FEATURE_COLUMNS),'categorical_features':[],'scale_numerical':bool(SELECTED_SPEC['scale']),'scaler':'StandardScaler' if SELECTED_SPEC['scale'] else None,'fit_scope':'training_partition_or_training_fold_only'}
readiness={'preparation_handoff_validated':True,'frozen_partitions_respected':True,'regression_cv_completed':True,'candidate_models_evaluated':True,'feature_policy_frozen':True,'selected_candidate_frozen':True,'regression_metric_contract_frozen':True,'model_selection_handoff_reloadable':True,'test_partition_sealed':True,'test_partition_evaluated':False,'final_model_training_ready':True,'final_model_trained':False,'model_artifact_materialized':False,'model_bundle_materialized':False,'operational_modeling_ready':False,'operational_validity':'unconfirmed'}
handoff={'schema_version':'model-selection-handoff.v3','artifact_type':'model_selection_handoff','dataset_slug':DATASET_SLUG,'problem_type':'continuous_regression','preparation_handoff_reference':manifest['preparation_handoff_reference'],'preparation_artifact_hashes':prep_hashes,'target_contract':target_contract,'available_feature_columns':list(FEATURE_COLUMNS),'selected_feature_columns':list(FEATURE_COLUMNS),'selected_feature_policy':'all_features','selected_model_id':SELECTED_MODEL_ID,'selected_model_family':SELECTED_SPEC['family'],'selected_hyperparameters':FROZEN_PARAMETERS[SELECTED_MODEL_ID],'selected_estimator_fixed_constructor_parameters':{k:v for k,v in SELECTED_SPEC['estimator'].get_params(deep=False).items() if v is not None},'selected_preprocessing_contract':preprocessing,'primary_metric':'mae','primary_metric_direction':'lower_is_better','secondary_metrics':['rmse','r2','medae'],'cv_contract':manifest['cv_contract'],'random_seeds':manifest['random_seeds'],'selected_cv_evidence':selected_summary,'selected_validation_evidence':SELECTED_VALIDATION['metrics'],'selection_rationale':SELECTION['selection_rationale'],'tie_break_rationale':{'tolerance_mpa':0.10,'criteria_applied':SELECTION['criteria_applied']},'analysis_conclusions':{'hypothesis':HYPOTHESIS,'sensitivities_used_for_selection':False},'test_partition_sealed':True,'test_partition_evaluated':False,'final_training_instructions':{'notebook':'notebooks/04_final_model_and_bundle.ipynb','reconstruct_pipeline_from_contract':True,'fit_partitions':['train','validation'],'final_evaluation_partition':'test','access_test_only_after_contract_freeze_and_final_fit':True,'evaluate_test_once':True,'do_not_retune':True,'do_not_change_feature_policy':True,'do_not_change_hyperparameters':True,'do_not_change_preprocessing':True,'target_scale':'original MPa scale','prediction_type':'continuous_numeric'},'final_model_training_ready':True,'final_model_trained':False,'model_artifact':None,'model_artifact_materialized':False,'bundle':None,'model_bundle_materialized':False,'operational_modeling_ready':False,'operational_validity':'unconfirmed','feature_inference_availability':'unconfirmed','readiness':readiness}
ARTIFACTS={'model-selection-manifest.json':manifest,'candidate-results.json':candidate_results,'cross-validation-results.csv':CV_RESULTS,'validation-evidence.json':validation_evidence,'selection-analysis.json':selection_analysis,'model-selection-handoff.json':handoff}
WRITE_RESULT=write_regression_model_selection_artifacts(output_directory=PROJECT.path(MODEL_SELECTION_ROOT),artifacts=ARTIFACTS,overwrite=False)


## 15. Model-Selection Handoff Validation

A separate Python process is used after this notebook run to prove independent v3 reload and hash verification. The contract contains everything needed to reconstruct the selected pipeline in an unfitted state.

In [15]:
reload_code = """import json
from scripts.select_models import load_and_validate_model_selection_handoff
p = load_and_validate_model_selection_handoff(project_root='.', handoff_path='artifacts/model-selection/concrete-compressive-strength/model-selection-handoff.json')
assert p['schema_version'] == 'model-selection-handoff.v3'
assert p['problem_type'] == 'continuous_regression'
assert p['selected_model_id'] == 'hist_gradient_boosting'
assert p['selected_feature_columns'] == ['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']
assert p['target_contract']['unit'] == 'MPa'
assert p['primary_metric'] == 'mae' and p['primary_metric_direction'] == 'lower_is_better'
assert p['test_partition_sealed'] is True and p['test_partition_evaluated'] is False
assert p['final_model_training_ready'] is True
print(json.dumps({'schema_version': p['schema_version'], 'selected_model_id': p['selected_model_id'], 'target_unit': p['target_contract']['unit'], 'test_partition_sealed': p['test_partition_sealed'], 'test_partition_evaluated': p['test_partition_evaluated'], 'final_model_training_ready': p['final_model_training_ready']}, sort_keys=True))
"""
reload_result = subprocess.run([sys.executable, '-c', reload_code], cwd=PROJECT.root, check=True, capture_output=True, text=True)
assert reload_result.returncode == 0
reload_evidence = json.loads(reload_result.stdout)
assert reload_evidence == {'final_model_training_ready': True, 'schema_version': 'model-selection-handoff.v3', 'selected_model_id': 'hist_gradient_boosting', 'target_unit': 'MPa', 'test_partition_evaluated': False, 'test_partition_sealed': True}
assert reload_result.stderr == ''
